# SaaS Subscription Analytics — Complete Solution

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
DATA=Path('../02_Data/Clean_Reference'); OUT=Path('../07_Solution_Outputs')
accounts=pd.read_csv(DATA/'dim_accounts.csv',parse_dates=['signup_date'])
monthly=pd.read_csv(DATA/'fact_monthly_account_snapshot.csv',parse_dates=['month'])
invoices=pd.read_csv(DATA/'fact_invoices.csv',parse_dates=['invoice_month','due_date','paid_date'])

In [ ]:
latest=monthly[monthly.month.eq(monthly.month.max())]
active=latest[latest.churned_flag.eq(0)]
print('Latest month:',monthly.month.max().date())
print('Active logos:',active.account_id.nunique())
print('MRR £:',round(active.mrr_gbp.sum(),2))
print('ARR £:',round(active.mrr_gbp.sum()*12,2))

## NRR / GRR trend

In [ ]:
k=pd.read_csv(OUT/'monthly_saas_kpis.csv',parse_dates=['month'])
display(k.tail(12))
k.set_index('month')[['nrr','grr']].tail(24).plot(figsize=(10,4),title='Revenue Retention'); plt.axhline(1,ls='--'); plt.show()

## Customer health

In [ ]:
health=pd.read_csv(OUT/'customer_health_scores.csv')
display(health.groupby('health_band').agg(accounts=('account_id','nunique'),ARR=('annual_arr_gbp','sum'),risk_ARR=('risk_arr_gbp','sum'),avg_health=('health_score','mean')).sort_values('avg_health'))

## Pricing leakage

In [ ]:
leak=pd.read_csv(OUT/'pricing_discount_leakage.csv')
print('Annualized excess-discount leakage £:',round(leak.estimated_annual_discount_leakage_gbp.sum(),2))
display(leak.head(10))

## Acquisition economics

In [ ]:
cac=pd.read_csv(OUT/'cac_by_channel_month.csv')
display(cac.groupby('acquisition_channel').agg(spend=('marketing_sales_spend_gbp','sum'),new_customers=('new_customers','sum')).assign(CAC=lambda d:d.spend/d.new_customers).sort_values('CAC'))

## Interpretable churn propensity

In [ ]:
last=monthly.sort_values('month').groupby('account_id').tail(1).copy()
pay=invoices.groupby('account_id').agg(failed=('payment_status',lambda s:(s=='Failed').sum()),late=('payment_status',lambda s:(s=='Late').sum())).reset_index()
f=last.merge(pay,on='account_id',how='left').fillna({'failed':0,'late':0})
cols=['seat_utilization','support_tickets','critical_tickets','avg_resolution_hours','discount_pct','failed','late','seats']
X=f[cols].fillna(f[cols].median()); y=f.churned_flag
sc=StandardScaler(); xs=sc.fit_transform(X)
model=LogisticRegression(max_iter=2000,class_weight='balanced',random_state=42).fit(xs,y)
prob=model.predict_proba(xs)[:,1]
print('AUC:',round(roc_auc_score(y,prob),3))
display(pd.DataFrame({'feature':cols,'coefficient':model.coef_[0]}).sort_values('coefficient',ascending=False))

## Expansion / renewal priority lists

In [ ]:
display(pd.read_csv(OUT/'expansion_candidates.csv').head(20))
display(pd.read_csv(OUT/'next_90_day_renewal_priorities.csv').head(20))

## Final management story
See `06_Consulting_Workflow/05_Executive_Story_Solution.md`.